# Chapter 1 — pandas 是什么，怎么用

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaosfrey-arch/news-classification-tutorial/blob/main/chapter01_pandas.ipynb)

**本章目标：** 用 pandas 加载并初步探索 dataset.csv。

**预计时间：** 45 分钟

## 1.1 数据在电脑里长什么样？

现实世界的数据往往是**表格**形式：一行一条记录，一列一个字段。

`dataset.csv` 就是这样一张表，共 128,600 行，3 列：

| Title | Description | Class |
|---|---|---|
| Vikings ink Morten Andersen | Eden Prairie, MN - The Minnesota Vikings... | Sports |
| Oil majors cash in as price hikes... | MAKE hay while the sun shines... | Business |
| ... | ... | ... |

**CSV（Comma-Separated Values）** = 逗号分隔的纯文本文件，每行是一条记录，列之间用逗号隔开。

## 1.2 pandas 是什么？

**pandas** 是 Python 里专门处理表格数据的库，可以理解为「Python 版的 Excel」，但功能更强、速度更快、适合处理大数据。

核心数据结构叫 **DataFrame**（数据框），就是一张带行号和列名的表格。

> 📖 官方文档：https://pandas.pydata.org/docs/

In [1]:
# 第一步：导入 pandas，约定俗成简称 pd
import pandas as pd

print('pandas 版本：', pd.__version__)

pandas 版本： 2.2.2


## 1.3 加载数据：read_csv()

In [3]:
# pd.read_csv() 读取 CSV 文件，返回一个 DataFrame
# 如果文件在 Colab 根目录，直接写文件名

# 下载 dataset.csv 文件
!wget -q https://raw.githubusercontent.com/chaosfrey-arch/news-classification-tutorial/main/dataset.csv

df = pd.read_csv('dataset.csv')

# 第一次加载可能需要几秒，请耐心等待
print('数据加载完成！')

数据加载完成！


## 1.4 查看数据：head() / tail() / shape

In [4]:
# head(n) 显示前 n 行，默认 5 行
# 运行后能看到表格结构
df.head()

,Title,Description,Class
0,Vikings ink Morten Andersen,"Eden Prairie, MN (Sports Network) - The Minnes...",Sports
1,Orioles Drub Streaking Yankees 14-8,BALTIMORE - Brian Roberts and Larry Bigbie had...,World
2,Oil majors cash in as price hikes mean soaring...,"MAKE hay while the sun shines, they say. Or, i...",Business
3,"UPDATE 3-Allied Waste again cuts forecast, sha...","Allied Waste Industries Inc. (AW.N: Quote, Pro...",Business
4,Future of Illinois Farm May Lie in Swampy Past,"Environmentalists say they can return a 7,000-...",Science/Tech


In [7]:
# shape 属性：返回 (行数, 列数)
a, b = df.shape
print(f'这个数据集共有 {a:,} 行，{b} 列')
print(f'也就是 {行数:,} 篇新闻文章，每篇有 {列数} 个字段')

这个数据集共有 128,600 行，3 列
也就是 128,600 篇新闻文章，每篇有 3 个字段


In [8]:
# columns 属性：所有列名
print('列名：', df.columns.tolist())

# dtypes 属性：每列的数据类型
print('\n各列数据类型：')
print(df.dtypes)

列名： ['Title', 'Description', 'Class']

各列数据类型：
Title          object
Description    object
Class          object
dtype: object


## 1.5 查看某一列

In [9]:
# df['列名'] 取出单独一列，返回 Series（序列）
# 看 Class 列的前 10 个值
print(df['Class'].head(10).tolist())

['Sports', 'World', 'Business', 'Business', 'Science/Tech', 'World', 'Science/Tech', 'Business', 'Business', 'Science/Tech']


In [10]:
# value_counts() 统计每个值出现的次数
# 这是最常用的操作之一！
print('各类别文章数量：')
print(df['Class'].value_counts())

# 转成百分比
print('\n各类别占比：')
print(df['Class'].value_counts(normalize=True).round(4) * 100, '%')

各类别文章数量：
Class
Science/Tech    32037
Sports          32028
World           32018
Business        32017
Name: count, dtype: int64

各类别占比：
Class
Science/Tech    25.01
Sports          25.00
World           24.99
Business        24.99
Name: proportion, dtype: float64 %


## 1.6 发现缺失值：isnull()

**缺失值（Missing Value）** = 表格里某个格子是空的，没有数据。

这很常见——也许记者没写描述，或者数据采集时出了问题。缺失值会导致程序报错，必须处理。

In [11]:
# isnull() 返回 True/False 的表格（True = 这格是空的）
# .sum() 统计每列有多少个空格
print('每列缺失值数量：')
print(df.isnull().sum())

print(f'\n总缺失值：{df.isnull().sum().sum()} 个')
print(f'占总数据的 {df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100:.2f}%')

每列缺失值数量：
Title            0
Description    250
Class          500
dtype: int64

总缺失值：750 个
占总数据的 0.19%


## 1.7 查看文本长度

In [12]:
# 计算 Title 的字符长度
df['title_len'] = df['Title'].str.len()

# describe() 给出统计摘要：最小值、最大值、平均值、分位数等
print('标题长度统计：')
print(df['title_len'].describe().round(1))

print(f'\n最短标题：{df["title_len"].min()} 个字符')
print(f'最长标题：{df["title_len"].max()} 个字符')
print(f'平均标题：{df["title_len"].mean():.0f} 个字符')

标题长度统计：
count    128600.0
mean         42.1
std          13.6
min           6.0
25%          33.0
50%          41.0
75%          49.0
max         115.0
Name: title_len, dtype: float64

最短标题：6 个字符
最长标题：115 个字符
平均标题：42 个字符


In [13]:
# 计算 Description 的词数（按空格切分）
# 注意：Description 有缺失值，要先填充空字符串
df['desc_words'] = df['Description'].fillna('').str.split().str.len()

print('描述词数统计：')
print(df['desc_words'].describe().round(1))
print(f'\n99% 的文章词数 ≤ {df["desc_words"].quantile(0.99):.0f} 个词')

描述词数统计：
count    128600.0
mean         31.0
std           9.9
min           0.0
25%          25.0
50%          30.0
75%          36.0
max         173.0
Name: desc_words, dtype: float64

99% 的文章词数 ≤ 63 个词


## 🏋️ 小练习

In [ ]:
# 练习 1：找出 Title 最长的那篇文章，打印出它的标题和类别
# 提示：用 df.loc[df['title_len'].idxmax()]

# 你的代码写在这里：


In [ ]:
# 练习 1：找出 Title 最长的那篇文章，打印出它的标题和类别
# 提示：用 df.loc[df['title_len'].idxmax()]

# 你的代码写在这里：
longest_title_article = df.loc[df['title_len'].idxmax()]
print(f"标题最长的文章是：\n标题: {longest_title_article['Title']}\n类别: {longest_title_article['Class']}")

In [15]:
longest_title_article=df.loc[df['title_len'].idxmax()]
print(f"longest:\ntitile: {longest_title_article["Title"]}\nClass:{longest_title_article["Class"]}")

longest:
titile: Attachmate Heightens Security, Centralises Management and Brings Microsoft Usability to Host Access with EXTRA! ...
Class:Science/Tech


In [ ]:
# 练习 2：计算 Business 类别文章的平均描述词数
# 提示：先用 df[df['Class']=='Business'] 筛选，再用 .mean()

# 你的代码写在这里：


## 总结

| 操作 | 代码 | 说明 |
|------|------|------|
| 读文件 | `pd.read_csv('file.csv')` | 返回 DataFrame |
| 看前几行 | `df.head(5)` | 快速预览 |
| 行列数 | `df.shape` | (行数, 列数) |
| 取一列 | `df['列名']` | 返回 Series |
| 统计值 | `df['列名'].value_counts()` | 每个值出现几次 |
| 缺失值 | `df.isnull().sum()` | 每列空值数量 |
| 统计摘要 | `df['列名'].describe()` | 均值/最大/最小等 |

**下一章 →** Chapter 2：用图说话：数据可视化